In [0]:
%run ../cralf_config/nb_00_config_lakeflow

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_enriched_orders = spark.table(enriched_orders_table)

In [0]:
df_silver = (
    df_enriched_orders
        .filter(col('order_date') == processing_date)
)

Customer Revenue Aggregation

In [0]:
df_customer_revenue = (
    df_silver
        .groupBy("user_id")
        .agg(
            sum("revenue").alias("total_revenue")
        )
)

Customer Segmentation

In [0]:
df_customer_segment = (
    df_customer_revenue
        .withColumn("segment",
                     when(col("total_revenue") > 1000, "High-Value")
                    .when( (col("total_revenue") > 500) & (col("total_revenue") <= 1000) , "Medium-Value")
                    .otherwise("Low-Value")
                    )
)

Product Revenue Aggregation

In [0]:
df_product_revenue = (
    df_silver
        .groupBy("product_id","category")
        .agg(
            sum("revenue").alias("total_revenue"),
            sum("quantity").alias("total_quantity")
        )
)

df_product_revenue.printSchema()

In [0]:
from pyspark.sql.window import *

Top Products

In [0]:
window_spec = Window.orderBy(col("total_revenue").desc())

df_top_products = (
    df_product_revenue
        .withColumn("rank", row_number().over(window_spec))
        .filter(col("rank") <= 10)
)

display(df_top_products)

In [0]:
df_customer_segment.write.format("delta")\
    .mode("append")\
    .saveAsTable(customer_revenue_table)

In [0]:
df_product_revenue.write.format("delta")\
    .mode("append")\
    .saveAsTable(product_revenue_table)

In [0]:
df_top_products.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.cralf_top_products")